In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
import os
import urllib.request
import tarfile
import time

# --- Hyperparameters ---
image_size = 224  # ResNet152 expects input images of size 224x224
batch_size = 32
num_epochs = 2# You can adjust the number of epochs
learning_rate = 0.001
momentum = 0.9
# ------------------------

# Check for GPU availability and set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Data Preparation
data_dir = "./dog_data"
if not os.path.exists(data_dir):
    os.makedirs(data_dir)

dataset_url = "http://vision.stanford.edu/aditya86/ImageNetDogs/images.tar"
data_file = os.path.join(data_dir, "images.tar")

if not os.path.exists(data_file):
    print("Downloading dataset...")
    urllib.request.urlretrieve(dataset_url, data_file)
    print("Dataset downloaded.")

    print("Extracting dataset...")
    with tarfile.open(data_file, "r") as tar_ref:
        tar_ref.extractall(data_dir)
    print("Dataset extracted.")
else:
    print("Dataset already exists.")

# Image transformations with data augmentation
data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(image_size),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(degrees=15),  # Added rotation
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),  # Added color jitter
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize(image_size),
        transforms.CenterCrop(image_size),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

# Load the datasets with ImageFolder
image_datasets = {x: datasets.ImageFolder(os.path.join(data_dir, 'Images'), data_transforms[x]) for x in ['train', 'val']}

# Get the class names before splitting
class_names = image_datasets['train'].classes

# Split dataset into train and validation (80% train, 20% validation)
train_size = int(0.6 * len(image_datasets['train']))
val_size = len(image_datasets['train']) - train_size
image_datasets['train'], image_datasets['val'] = random_split(image_datasets['train'], [train_size, val_size])

# Create DataLoaders
dataloaders = {x: DataLoader(image_datasets[x], batch_size=batch_size, shuffle=True if x == 'train' else False, num_workers=0) for x in ['train', 'val']}  # num_workers=0 for CPU
dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val']}


# Model Definition (ResNet152)
model = models.resnet152(pretrained=True)

# Modify the classifier part of the model
num_ftrs = model.fc.in_features  # Get the input features of the last layer
model.fc = nn.Linear(num_ftrs, len(class_names))  # Replace the last layer

model = model.to(device)

# Loss Function and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=learning_rate, momentum=momentum)

# Training Loop with timing
start_time = time.time()

for epoch in range(num_epochs):
    print(f'Epoch {epoch + 1}/{num_epochs}')
    print('-' * 10)

    for phase in ['train', 'val']:
        if phase == 'train':
            model.train()  # Set model to training mode
        else:
            model.eval()   # Set model to evaluate mode

        running_loss = 0.0
        running_corrects = 0

        for inputs, labels in dataloaders[phase]:
            inputs = inputs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            with torch.set_grad_enabled(phase == 'train'):
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)

                if phase == 'train':
                    loss.backward()
                    optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)

        epoch_loss = running_loss / dataset_sizes[phase]
        epoch_acc = running_corrects.double() / dataset_sizes[phase]

        print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

end_time = time.time()
execution_time = end_time - start_time
print(f"Training complete in {execution_time:.2f} seconds")

# Save the trained model
torch.save(model.state_dict(), 'dog_breed_classifier_resnet152.pth')  # Save with a different name
print('Model saved.')

Using device: cuda
Dataset downloaded.
Extracting dataset...


/tmp/ipykernel_218/1449533292.py:38: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar_ref.extractall(data_dir)


Dataset extracted.


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet152_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet152_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet152-394f9c45.pth" to /root/.cache/torch/hub/checkpoints/resnet152-394f9c45.pth


100%|██████████| 230M/230M [00:01<00:00, 191MB/s]


Epoch 1/2
----------
train Loss: 3.1918 Acc: 0.3887
val Loss: 1.6368 Acc: 0.6452
Epoch 2/2
----------
train Loss: 1.4986 Acc: 0.6633
val Loss: 1.1392 Acc: 0.7074
Training complete in 1067.39 seconds
Model saved.


In [ ]:
import torch
import torch.nn as nn
from torchvision import models, transforms, datasets
from PIL import Image
from google.colab import files  # If using Google Colab
import os

# 1. Load the trained ResNet152 model
model = models.resnet152(pretrained=False)  # Load ResNet152 without pretrained weights

# Get the number of classes from the saved model's state_dict
state_dict = torch.load('dog_breed_classifier_resnet152.pth')  # Path to your saved model
num_classes = state_dict['fc.weight'].shape[0]  # Get number of classes from weight shape

# Modify the classifier part of the model to have the correct number of classes
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, num_classes)

# Now load the state_dict
model.load_state_dict(state_dict)
model.eval()  # Set the model to evaluation mode

# Before making predictions, define class_names
# If you saved class_names during training, load them here.
# If not, recreate them using the dataset path from your training script:

# Assuming data_dir is the same as in your training script:
data_dir = "./dog_data"
class_names = datasets.ImageFolder(os.path.join(data_dir, 'Images')).classes



# 2. Define the image transformation (same as used during training for ResNet152)
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),  # ResNet expects 224x224 images
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# 3. Upload the image (if using Google Colab)
uploaded = files.upload()
image_path = list(uploaded.keys())[0]

# 4. Preprocess the image
image = Image.open(image_path)
image_tensor = transform(image).unsqueeze(0)  # Add batch dimension

# 5. Make the prediction
with torch.no_grad():  # Disable gradient calculation during inference
    outputs = model(image_tensor)
    _, predicted_class = torch.max(outputs, 1)  # Get the class with highest probability

# 6. Get the predicted class name
predicted_label = class_names[predicted_class.item()]
print(f"Predicted Class: {predicted_label}")

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Saving dog to dog
Predicted Class: n02106030-collie


In [ ]:
from google.colab import files
files.download('dog_breed_classifier_resnet152.pth')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>